# LIWC22-style Toxicity Analysis

This notebook measures toxic language in Facebook ads from the 2022 Australian federal election study.

It uses the same processed dataset as the other analysis notebooks and always filters to the main parties used in `party_analysis.ipynb`.


In [ ]:
%load_ext watermark
%watermark -v -n -m -p numpy,pandas,scipy,sklearn,matplotlib,seaborn

import warnings
import os
import sys
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2
%matplotlib inline

print(f'default sys.path: {sys.path}')
PROJ_ROOT = os.path.abspath(os.path.join(os.pardir))
PROJ_ROOT = os.path.abspath(os.path.join(PROJ_ROOT, os.pardir))
if PROJ_ROOT not in sys.path:
    sys.path.append(PROJ_ROOT)
print(f'Project root: {PROJ_ROOT}')

from utils import mpl_settings
from utils.dataset_utilities import load_data
from utils import config
from IPython.display import display

mpl_settings.apply_settings()
plt.style.use(mpl_settings.plot_settings)

FILE_PATH = '../../data/processed/2022_aus_elections_mar_to_may.csv'
df = load_data(FILE_PATH).copy()

main_parties = ['Liberal', 'Labor', 'Green', 'Independent', 'United Australia Party']
df = df[df['macro_party_uap'].isin(main_parties)].copy()

for col in ['ad_delivery_start_time', 'ad_delivery_stop_time']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

for col in ['ad_creative_body', 'multiword_safe_lemmatized', 'stemmed_body']:
    if col not in df.columns:
        df[col] = np.nan

# Use the richest available text column, but keep the original body text when present.
df['text_for_toxicity'] = (
    df['ad_creative_body']
      .fillna(df['multiword_safe_lemmatized'])
      .fillna(df['stemmed_body'])
      .fillna('')
      .astype(str)
      .str.strip()
)

# Basic timing and length features for normalization.
df['word_count'] = df['text_for_toxicity'].str.findall(r"[A-Za-z']+").str.len().fillna(0).astype(int)
df['duration_days'] = (
    (df['ad_delivery_stop_time'].fillna(df['ad_delivery_start_time']) - df['ad_delivery_start_time']).dt.days + 1
).fillna(1)
df['duration_days'] = df['duration_days'].clip(lower=1).astype(int)
df['impressions_per_day'] = df['mean_impressions'] / df['duration_days']
df['week_start'] = df['ad_delivery_start_time'].dt.to_period('W-SUN').dt.start_time

print(f'Rows after main-party filter: {len(df):,}')
display(df[['macro_party_uap', 'mean_spend', 'mean_impressions', 'duration_days', 'word_count']].describe(include='all'))


## LIWC22 Loader

The notebook looks for a LIWC22 dictionary under `../../data/external/`. If no file is found, it falls back to an explicit toxic-word lexicon so the notebook remains runnable.


In [ ]:
WORD_RE = re.compile(r"[A-Za-z']+")

LIWC_CANDIDATES = [
    Path('../../data/external/LIWC22.dic'),
    Path('../../data/external/LIWC-22.dic'),
    Path('../../data/external/liwc22.dic'),
    Path('../../data/external/liwc22.txt'),
    Path('../../data/external/LIWC22.txt'),
]

# Heuristics for selecting toxicity-relevant LIWC categories when a dictionary is available.
LIWC_CATEGORY_HINTS = [
    'neg', 'negative', 'anger', 'swear', 'profan', 'viol', 'attack', 'aggress',
    'hostil', 'hate', 'insult', 'threat', 'decept', 'dishon', 'death', 'toxic', 'risk'
]

PLACEHOLDER_LEXICON = {
    'negative': [
        'bad', 'worst', 'awful', 'terrible', 'horrible', 'disgusting', 'shame', 'shameful',
        'failure', 'fraud', 'corrupt', 'corruption', 'scam', 'deceive', 'dishonest', 'liar',
        'lie', 'lies', 'lying', 'broken', 'damaging', 'dangerous', 'toxic', 'failing', 'failed'
    ],
    'aggression': [
        'attack', 'attacks', 'attacking', 'fight', 'fighting', 'destroy', 'destroying', 'hurt',
        'hurting', 'kill', 'killing', 'violent', 'violence', 'threat', 'threaten', 'threatening',
        'enemy', 'enemies', 'war', 'hostile', 'aggressive'
    ],
    'swear': [
        'fuck', 'shit', 'bullshit', 'asshole', 'bastard', 'crap', 'damn'
    ],
    'deception': [
        'fake', 'false', 'lie', 'lies', 'lying', 'deceive', 'deceptive', 'misleading', 'hoax', 'scam'
    ],
}


def slugify(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return text.strip('_') or 'category'


def term_to_regex(term):
    # LIWC wildcards are prefix-style (`*`) so we map them to a regex word fragment.
    cleaned = str(term).strip().lower()
    escaped = re.escape(cleaned)
    escaped = escaped.replace(r'\*', r'\w*')
    escaped = escaped.replace(r'\ ', r'\s+')
    return re.compile(r'(?<!\w)' + escaped + r'(?!\w)', flags=re.IGNORECASE)


def parse_liwc_dictionary(path):
    """Parse a LIWC-style dictionary file with category definitions followed by term mappings."""
    category_map = {}
    term_map = defaultdict(list)
    in_term_section = False

    with open(path, 'r', encoding='utf-8', errors='ignore') as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line or line.startswith('#'):
                continue
            if line == '%':
                in_term_section = True
                continue

            parts = re.split(r'\s+', line)
            if not in_term_section:
                if len(parts) >= 2 and parts[0].isdigit():
                    category_map[int(parts[0])] = ' '.join(parts[1:])
                continue

            if len(parts) >= 2:
                term = parts[0].lower()
                category_ids = [int(part) for part in parts[1:] if part.isdigit()]
                if category_ids:
                    term_map[term].extend(category_ids)

    selected_category_ids = {
        cat_id: name
        for cat_id, name in category_map.items()
        if any(hint in name.lower() for hint in LIWC_CATEGORY_HINTS)
    }

    selected_terms = defaultdict(set)
    for term, cat_ids in term_map.items():
        for cat_id in cat_ids:
            if cat_id in selected_category_ids:
                selected_terms[selected_category_ids[cat_id]].add(term)

    category_terms = {name: sorted(terms) for name, terms in selected_terms.items() if terms}
    compiled_patterns = {
        category: [term_to_regex(term) for term in terms]
        for category, terms in category_terms.items()
    }

    return {
        'source': str(path),
        'category_map': category_map,
        'selected_category_ids': selected_category_ids,
        'category_terms': category_terms,
        'compiled_patterns': compiled_patterns,
    }


def build_placeholder_resource():
    category_terms = {name: sorted(set(terms)) for name, terms in PLACEHOLDER_LEXICON.items()}
    compiled_patterns = {
        category: [term_to_regex(term) for term in terms]
        for category, terms in category_terms.items()
    }
    return {
        'source': 'placeholder',
        'category_map': {},
        'selected_category_ids': {},
        'category_terms': category_terms,
        'compiled_patterns': compiled_patterns,
    }


liwc_resource = None
for candidate in LIWC_CANDIDATES:
    if candidate.exists():
        try:
            parsed = parse_liwc_dictionary(candidate)
            if parsed['category_terms']:
                liwc_resource = parsed
                print(f'Loaded LIWC-style dictionary from: {candidate}')
                break
            print(f'Found {candidate}, but no toxicity-relevant categories were detected there.')
        except Exception as exc:
            print(f'Failed to parse {candidate}: {exc}')

if liwc_resource is None:
    liwc_resource = build_placeholder_resource()
    print('No LIWC22 dictionary file was found or parsed successfully.')
    print('Place a LIWC22 dictionary under one of these paths and rerun the notebook:')
    for candidate in LIWC_CANDIDATES:
        print(f' - {candidate}')
    print('Using a placeholder toxic-word lexicon for now.')

category_columns = {category: f'liwc_{slugify(category)}' for category in liwc_resource['category_terms']}
print('Selected toxicity-relevant categories:')
for category, terms in liwc_resource['category_terms'].items():
    print(f' - {category}: {len(terms)} terms')


In [ ]:
def count_matches(text, patterns):
    if not isinstance(text, str):
        text = ''
    text = text.lower()
    return sum(len(pattern.findall(text)) for pattern in patterns)


def score_toxicity(text, resource):
    if not isinstance(text, str):
        text = ''
    normalized = text.lower()
    tokens = WORD_RE.findall(normalized)
    word_count = len(tokens)

    scores = {}
    total = 0
    for category, patterns in resource['compiled_patterns'].items():
        count = count_matches(normalized, patterns)
        scores[f'liwc_{slugify(category)}'] = count
        total += count

    scores['toxicity_count'] = total
    scores['toxicity_flag'] = int(total > 0)
    scores['toxicity_rate_per_100_words'] = (100.0 * total / word_count) if word_count else 0.0
    scores['matched_words_per_100_words'] = scores['toxicity_rate_per_100_words']
    scores['word_count_for_matching'] = word_count
    return scores


score_records = [score_toxicity(text, liwc_resource) for text in df['text_for_toxicity'].tolist()]
toxicity_scores = pd.DataFrame.from_records(score_records)
toxicity_df = pd.concat([df.reset_index(drop=True), toxicity_scores.reset_index(drop=True)], axis=1)

# Weighted versions used later for comparison by party.
def weighted_mean(values, weights):
    values = pd.Series(values, dtype='float64')
    weights = pd.Series(weights, dtype='float64').fillna(0)
    mask = values.notna() & weights.notna()
    values = values[mask]
    weights = weights[mask]
    weight_sum = weights.sum()
    if weight_sum <= 0:
        return np.nan
    return float(np.average(values, weights=weights))


def build_party_summary(frame):
    rows = []
    for party, group in frame.groupby('macro_party_uap'):
        row = {
            'party': party,
            'ads': len(group),
            'mean_toxicity_rate_per_100_words': group['toxicity_rate_per_100_words'].mean(),
            'median_toxicity_rate_per_100_words': group['toxicity_rate_per_100_words'].median(),
            'share_toxic_ads': group['toxicity_flag'].mean(),
            'mean_toxicity_count': group['toxicity_count'].mean(),
            'impression_weighted_toxicity_rate': weighted_mean(group['toxicity_rate_per_100_words'], group['mean_impressions']),
            'spend_weighted_toxicity_rate': weighted_mean(group['toxicity_rate_per_100_words'], group['mean_spend']),
            'mean_spend': group['mean_spend'].mean(),
            'mean_impressions': group['mean_impressions'].mean(),
        }
        rows.append(row)
    summary = pd.DataFrame(rows).sort_values('mean_toxicity_rate_per_100_words', ascending=False)
    return summary


party_summary = build_party_summary(toxicity_df)
display(party_summary)

print('Overall toxicity statistics:')
display(toxicity_df[['toxicity_count', 'toxicity_rate_per_100_words', 'toxicity_flag']].describe())

# A simple association test for whether toxicity incidence differs by party.
contingency = pd.crosstab(toxicity_df['macro_party_uap'], toxicity_df['toxicity_flag'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f'Chi-square test on toxicity_flag by party: chi2={chi2:.3f}, dof={dof}, p={p_value:.4g}')


## Party Comparison

The first view compares raw and weighted toxicity rates by party. The weighted versions use impressions and spend as weights to show whether toxic language is concentrated in higher-reach ads.


In [ ]:
party_plot = party_summary.melt(
    id_vars='party',
    value_vars=['mean_toxicity_rate_per_100_words', 'impression_weighted_toxicity_rate', 'spend_weighted_toxicity_rate'],
    var_name='metric',
    value_name='toxicity_rate'
)

metric_labels = {
    'mean_toxicity_rate_per_100_words': 'Mean rate',
    'impression_weighted_toxicity_rate': 'Impression-weighted rate',
    'spend_weighted_toxicity_rate': 'Spend-weighted rate',
}
party_plot['metric'] = party_plot['metric'].map(metric_labels)

plt.figure(figsize=(14, 7))
sns.barplot(data=party_plot, x='party', y='toxicity_rate', hue='metric')
plt.ylabel('Toxic matches per 100 words')
plt.xlabel('Party')
plt.title('Toxicity by party')
plt.xticks(rotation=20, ha='right')
plt.legend(title='Metric', frameon=True)
plt.tight_layout()
plt.show()

print('Top ads by toxicity rate:')
display(
    toxicity_df.sort_values(['toxicity_rate_per_100_words', 'toxicity_count'], ascending=False)
    [['macro_party_uap', 'page_name', 'funding_entity', 'toxicity_count', 'toxicity_rate_per_100_words', 'mean_spend', 'mean_impressions', 'text_for_toxicity']]
    .head(10)
)


## Time Trend

This view tracks weekly average toxicity for each party. The main question is whether toxic language spikes in the run-up to election day or differs systematically across parties.


In [ ]:
weekly_summary = (
    toxicity_df
    .dropna(subset=['week_start'])
    .groupby(['week_start', 'macro_party_uap'])
    .agg(
        mean_toxicity_rate_per_100_words=('toxicity_rate_per_100_words', 'mean'),
        toxic_share=('toxicity_flag', 'mean'),
        ads=('toxicity_flag', 'size'),
        mean_impressions=('mean_impressions', 'mean'),
        mean_spend=('mean_spend', 'mean'),
    )
    .reset_index()
)

weekly_summary['rolling_mean_toxicity'] = (
    weekly_summary.sort_values('week_start')
    .groupby('macro_party_uap')['mean_toxicity_rate_per_100_words']
    .transform(lambda s: s.rolling(window=3, min_periods=1).mean())
)

plt.figure(figsize=(14, 7))
sns.lineplot(
    data=weekly_summary,
    x='week_start',
    y='mean_toxicity_rate_per_100_words',
    hue='macro_party_uap',
    marker='o'
)
plt.ylabel('Weekly mean toxic matches per 100 words')
plt.xlabel('Week')
plt.title('Weekly toxicity trend by party')
plt.xticks(rotation=20, ha='right')
plt.legend(title='Party', frameon=True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
sns.lineplot(
    data=weekly_summary,
    x='week_start',
    y='rolling_mean_toxicity',
    hue='macro_party_uap',
    marker='o'
)
plt.ylabel('3-week rolling mean toxic matches per 100 words')
plt.xlabel('Week')
plt.title('Smoothed toxicity trend by party')
plt.xticks(rotation=20, ha='right')
plt.legend(title='Party', frameon=True)
plt.tight_layout()
plt.show()


## Category Heatmap

When a LIWC-style dictionary is available, the notebook also reports category-level averages for the toxicity-relevant categories that were detected.


In [ ]:
category_cols = [f'liwc_{slugify(category)}' for category in liwc_resource['category_terms'].keys()]
category_lookup = {f'liwc_{slugify(category)}': category for category in liwc_resource['category_terms'].keys()}

if category_cols:
    category_party = toxicity_df.groupby('macro_party_uap')[category_cols].mean().rename(columns=category_lookup)
    plt.figure(figsize=(12, max(4, 0.6 * len(category_party))))
    sns.heatmap(category_party, annot=True, fmt='.3f', cmap='Reds', linewidths=0.5)
    plt.title('Average LIWC toxicity-category rates by party')
    plt.xlabel('Category')
    plt.ylabel('Party')
    plt.tight_layout()
    plt.show()
else:
    print('No toxicity categories were available for a heatmap.')


## LIWC22 Input Format And Plug-In Notes

Expected format for a LIWC-style dictionary file:

- Category definitions appear first, before the `%` separator.
- Term-to-category mappings appear after the `%` separator.
- Wildcards are supported with `*` and are treated as prefix matches in this notebook.

To plug in a real LIWC22 file later:

1. Place the dictionary file under `../../data/external/`.
2. Keep one of the candidate names used in the loader, or add your filename to `LIWC_CANDIDATES`.
3. Rerun the notebook. If the dictionary contains toxicity-relevant categories, the notebook will use them automatically.

If no dictionary is present, the placeholder lexicon is used so the rest of the analysis still runs.


## Short Takeaway

This final cell summarizes the highest-toxicity party under raw and weighted views. If the ranking changes after weighting, that suggests toxic language is distributed differently across reach and spend.


In [ ]:
top_raw = party_summary.sort_values('mean_toxicity_rate_per_100_words', ascending=False).iloc[0]
top_impression = party_summary.sort_values('impression_weighted_toxicity_rate', ascending=False).iloc[0]
top_spend = party_summary.sort_values('spend_weighted_toxicity_rate', ascending=False).iloc[0]

print(
    f"Highest raw toxicity rate: {top_raw['party']} "
    f"({top_raw['mean_toxicity_rate_per_100_words']:.3f} toxic matches per 100 words)"
)
print(
    f"Highest impression-weighted toxicity rate: {top_impression['party']} "
    f"({top_impression['impression_weighted_toxicity_rate']:.3f})"
)
print(
    f"Highest spend-weighted toxicity rate: {top_spend['party']} "
    f"({top_spend['spend_weighted_toxicity_rate']:.3f})"
)

if top_raw['party'] == top_impression['party'] == top_spend['party']:
    print('Weighting does not change the toxicity leader in this sample.')
else:
    print('Weighting changes the ranking, so toxicity is not distributed uniformly across reach/spend.')


In [ ]:
print('Notebook complete.')
print('Toxicity source:', liwc_resource['source'])
print('Categories used:', list(liwc_resource['category_terms'].keys()))
